# Hotel Reviews — BERTopic Pipeline

**Inputs** : `dataset-prepare/final-reviews-en.csv` / `final-reviews-vi.csv`  
**Embedding** : `paraphrase-multilingual-mpnet-base-v2` → saved to `embeddings_en.npy` / `embeddings_vi.npy`  
**Qdrant** : two separate collections — `hotel_reviews_en` / `hotel_reviews_vi`  
**Key idea** : embed once, extract any subset by DataFrame index → `embeddings_all[df.index]`

---
### Prerequisites
```bash
# 1. Run dataset-prepare/merge-and-preprocess.ipynb first
#    → produces final-reviews-en.csv and final-reviews-vi.csv

# 2. Start Qdrant
docker run -p 6333:6333 -p 6334:6334 \
    -v $(pwd)/qdrant_storage:/qdrant/storage \
    qdrant/qdrant
```

## Section 1 — Imports & Config

In [1]:
import sys
import random
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from topic_modeling import (
    EmbeddingEngine,
    QdrantStore,
    build_bertopic,
)

# ── Config ────────────────────────────────────────────────────────────────
LANGUAGE     = "en"          # "en" or "vi"
SAMPLE_SIZE  = 10_000        # set None to use all docs
RANDOM_SEED  = 42
NR_TOPICS    = "auto"
QDRANT_HOST  = "localhost"
QDRANT_PORT  = 6333

# min_cluster_size: smaller for vi, larger for en
MIN_CLUSTER_SIZE = 10 if LANGUAGE == "vi" else 50

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR     = Path("../dataset-prepare")
CSV_PATH     = DATA_DIR / f"final-reviews-{LANGUAGE}.csv"
EMB_PATH     = Path(f"./embeddings_{LANGUAGE}.npy")
COLLECTION   = f"hotel_reviews_{LANGUAGE}"
MODEL_DIR    = Path(f"./models/bertopic_model_{LANGUAGE}")

print(f"Language         : '{LANGUAGE}'")
print(f"CSV              : {CSV_PATH}  (exists={CSV_PATH.exists()})")
print(f"Embeddings cache : {EMB_PATH}  (exists={EMB_PATH.exists()})")
print(f"Qdrant collection: {COLLECTION}")
print(f"min_cluster_size : {MIN_CLUSTER_SIZE}")

Language         : 'en'
CSV              : ..\dataset-prepare\final-reviews-en.csv  (exists=True)
Embeddings cache : embeddings_en.npy  (exists=False)
Qdrant collection: hotel_reviews_en
min_cluster_size : 50


## Section 2 — Load Full Dataset

In [6]:
df_all = pd.read_csv(CSV_PATH, encoding="utf-8-sig", low_memory=False)
# Reset to a clean 0-based integer index — required for numpy indexing below
df_all = df_all.reset_index(drop=True)

print(f"Loaded {len(df_all):,} rows  |  columns: {df_all.columns.tolist()}")
df_all.head(2)

Loaded 135,326 rows  |  columns: ['review_text', 'language', 'rating', 'review_year', 'review_month', 'review_period', 'stay_nights', 'hotel_id', 'hotel_name', 'source', 'processed_text']


,review_text,language,rating,review_year,review_month,review_period,stay_nights,hotel_id,hotel_name,source,processed_text
0,A tourist class hotel with very basic amenitie...,en,3.2,2024.0,1.0,2024-01,4.0,163,Ramana Saigon Hotel,agoda,a tourist class hotel with very basic amenitie...
1,The hotel rooms etc are good but very very sti...,en,3.8,2024.0,1.0,2024-01,2.0,163,Ramana Saigon Hotel,agoda,the hotel rooms etc are good but very very sti...


## Section 3 — Embed & Store  *(run once per language)*

Embeds `processed_text` (already normalised/segmented by `preprocessor.py`).  
Saves to `embeddings_{lang}.npy` and upserts to Qdrant — **skipped if both already exist**.

In [7]:
engine = EmbeddingEngine(batch_size=64)

[EmbeddingEngine] Loading model: paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
if EMB_PATH.exists():
    print(f"Loading cached embeddings from {EMB_PATH} …")
    embeddings_all = np.load(EMB_PATH)
else:
    print("No cache found — encoding now …")
    docs_for_embed = [str(t) for t in df_all["processed_text"].tolist()]
    embeddings_all = engine.encode(docs_for_embed)
    np.save(EMB_PATH, embeddings_all)
    print(f"Saved → {EMB_PATH}")

print(f"embeddings_all : {embeddings_all.shape}  dtype={embeddings_all.dtype}")
assert len(embeddings_all) == len(df_all), "Embedding / DataFrame length mismatch!"

No cache found — encoding now …
[EmbeddingEngine] Encoding 135,326 texts …


Batches:   0%|          | 0/2115 [00:00<?, ?it/s]

[EmbeddingEngine] Done. Shape: (135326, 768)
Saved → embeddings_en.npy
embeddings_all : (135326, 768)  dtype=float32


In [10]:
# ── Qdrant upsert (skip if collection already populated) ──────────────────
store = QdrantStore(host=QDRANT_HOST, port=QDRANT_PORT, collection=COLLECTION)
store.create_collection(recreate=False)

if store.collection_count() == 0:
    print("Upserting to Qdrant …")
    store.upsert(df_all, embeddings_all)
else:
    print(f"Qdrant collection '{COLLECTION}' already has {store.collection_count():,} points — skipping upsert.")

[QdrantStore] Connected to Qdrant at localhost:6333  collection='hotel_reviews_en'
[QdrantStore] Created collection 'hotel_reviews_en' (dim=768).
Upserting to Qdrant …
[QdrantStore] Upserting 135,326 points in batches of 256 …
[QdrantStore] Upsert complete. Total points: 135,326


## Section 4 — Extract Docs & Embeddings

Filter `df_all` by any criteria, then pull the matching embeddings with `embeddings_all[df.index]`.  
Optionally downsample with `random.sample`.

In [12]:
# ── Filtering examples — uncomment / combine as needed ────────────────────

df = df_all.copy()

# Filter by source
# df = df[df["source"] == "agoda"]

# Filter by hotel_id
# df = df[df["hotel_id"].isin([9195, 902])]

# Filter by review period
df = df[df["review_year"] >= 2025]

# Filter by rating
# df = df[df["rating"] >= 4]

print(f"Filtered to {len(df):,} rows")

Filtered to 32,761 rows


In [ ]:
# ── Extract aligned embeddings using df.index ──────────────────────────────
docs       = [str(t) for t in df["processed_text"].tolist()]
embeddings = embeddings_all[df.index]

print(f"docs       : {len(docs):,}")
print(f"embeddings : {embeddings.shape}")

docs       : 32,761
embeddings : (32761, 768)


In [14]:
# # ── Optional random sample ─────────────────────────────────────────────────
# if SAMPLE_SIZE and len(docs) > SAMPLE_SIZE:
#     random.seed(RANDOM_SEED)
#     idx        = random.sample(range(len(docs)), SAMPLE_SIZE)
#     docs       = [docs[i] for i in idx]
#     embeddings = embeddings[idx]
#     df         = df.iloc[idx].reset_index(drop=True)
#     print(f"Sampled {SAMPLE_SIZE:,} from {len(df_all):,} docs.")

# print(f"Final — docs: {len(docs):,}  |  embeddings: {embeddings.shape}")
# print(f"Sample doc  : {docs[0][:120]}")

## Section 5 — Fit BERTopic

Pre-computed `embeddings` passed directly — UMAP/HDBSCAN use them, no re-encoding.

In [17]:
topic_model = build_bertopic(
    nr_topics=NR_TOPICS,
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_topic_size=MIN_CLUSTER_SIZE,
    embedding_model=engine.model,   # needed by KeyBERTInspired
)
topics, probs = topic_model.fit_transform(docs, embeddings)

n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
n_outlier = sum(1 for t in topics if t == -1)
print(f"\nLanguage     : '{LANGUAGE}'")
print(f"Topics found : {n_topics}")
print(f"Outliers     : {n_outlier:,} ({n_outlier / len(topics) * 100:.1f}%)")

[build_bertopic] BERTopic configured.


ValueError: Make sure that the embeddings are a numpy array with shape: (len(docs), vector_dim) where vector_dim is the dimensionality of the vector embeddings. 

## Section 6 — Inspect Topics

In [ ]:
topic_info = topic_model.get_topic_info()
print(f"Shape: {topic_info.shape}")
topic_info.head(10)

In [ ]:
# Top keywords for a specific topic
TOPIC_ID = 0
print(f"Topic {TOPIC_ID} keywords:")
for word, score in topic_model.get_topic(TOPIC_ID):
    print(f"  {word:<30} {score:.4f}")

## Section 7 — (Optional) Reduce Topics

In [ ]:
# TARGET = 25
# topic_model.reduce_topics(docs, nr_topics=TARGET)
# topics = topic_model.topics_
# print(f"After reduction: {len(set(topics)) - (1 if -1 in topics else 0)} topics")

## Section 8 — Visualizations

In [ ]:
# Intertopic distance map
topic_model.visualize_topics()

In [ ]:
# Top-N words per topic
topic_model.visualize_barchart(top_n_topics=12, n_words=8)

In [ ]:
# Topic hierarchy
topic_model.visualize_hierarchy()

In [ ]:
# Document UMAP scatter — sample for speed
n_vis = min(3_000, len(docs))
vis_idx = random.sample(range(len(docs)), n_vis)
topic_model.visualize_documents(
    [docs[i] for i in vis_idx],
    embeddings=embeddings[vis_idx],
    hide_document_hover=True,
)

## Section 9 — Topic Prominence

In [ ]:
print(f"Probability matrix shape: {probs.shape}")

topic_prominence = pd.DataFrame({
    "topic"    : list(range(probs.shape[1])),
    "mean_prob": probs.mean(axis=0),
}).sort_values("mean_prob", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(topic_prominence["topic"].astype(str), topic_prominence["mean_prob"])
ax.set_xlabel("Topic")
ax.set_ylabel("Mean probability")
ax.set_title(f"Topic Prominence [{LANGUAGE.upper()}]")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## Section 10 — Per-Hotel Topic Distribution

In [ ]:
# Attach topics back to the (sampled) df
df_result = df.copy()
df_result["topic"]    = topics
df_result["top_prob"] = probs.max(axis=1) if probs.ndim == 2 else probs

print(f"Language : '{LANGUAGE}'  |  rows: {len(df_result):,}")
df_result[["hotel_id", "hotel_name", "source", "review_text", "topic", "top_prob"]].head(5)

In [ ]:
HOTEL_ID     = df_result["hotel_id"].iloc[0]    # change to any hotel_id
hotel_df     = df_result[df_result["hotel_id"] == HOTEL_ID]
topic_counts = hotel_df["topic"].value_counts().drop(-1, errors="ignore")

print(f"Hotel {HOTEL_ID} [{LANGUAGE}] — {len(hotel_df)} reviews across {len(topic_counts)} topics")

fig, ax = plt.subplots(figsize=(8, 4))
topic_counts.plot(kind="bar", ax=ax)
ax.set_xlabel("Topic")
ax.set_ylabel("Review count")
ax.set_title(f"Topic distribution — Hotel {HOTEL_ID}  [{LANGUAGE.upper()}]")
plt.tight_layout()
plt.show()

## Section 11 — Save Model

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
topic_model.save(str(MODEL_DIR), serialization="safetensors", save_ctfidf=True)
print(f"Model saved → {MODEL_DIR.resolve()}")

# Load back:
# from bertopic import BERTopic
# topic_model = BERTopic.load(str(MODEL_DIR))